In [1]:
include("LiPoSID.jl")
using QuantumOptics
basis = NLevelBasis(2)
using LinearAlgebra
using HDF5
using Dates
using Statistics

In [2]:
# Compatibility fix: newer DiffEqBase removed promote_dual
import DiffEqBase
if !isdefined(DiffEqBase, :promote_dual)
    @eval DiffEqBase promote_dual(T, ::Type{S}) where {S} = T
end

promote_dual (generic function with 1 method)

In [3]:
σˣ = [ 0 1
       1 0 ]
σʸ = [ 0.   im*1
      -im*1  0   ]
σᶻ = [ 1.  0
       0  -1 ]

fᴷ₁ = σˣ/2;  fᴷ₂ = σʸ/2;  fᴷ₃ = σᶻ/2
fᴷᴼᴺᴮ = [fᴷ₁, fᴷ₂, fᴷ₃]

3-element Vector{Matrix{ComplexF64}}:
 [0.0 + 0.0im 0.5 + 0.0im; 0.5 + 0.0im 0.0 + 0.0im]
 [0.0 + 0.0im 0.0 + 0.5im; 0.0 - 0.5im 0.0 + 0.0im]
 [0.5 + 0.0im 0.0 + 0.0im; 0.0 + 0.0im -0.5 + 0.0im]

In [4]:
γ = ["0.079477","0.25133","0.79477","2.5133","7.9477","25.133","79.477","251.33"]
all_states = vcat(["B$i" for i in 1:4], ["D$i" for i in 1:10])   # 14 states
evol_data  = "DATA/ALL_GAMMAS_B4_D10.h5"

kossak_fitted   = "E_KOSSAK_CONSTR_TSSOS_treshold_1e-15_FROB_QO_2026-Jun-14_at_16-44.h5"
lindblad_fitted = "E_LINDBLAD4_CONSTR_TSSOS_treshold_1e-9_FROB_QO_2026-Jun-14_at_17-08.h5"

date_str = Dates.format(today(), "yyyy-mm-dd")

"2026-07-12"

In [5]:
# ── Kossakowski rescore ────────────────────────────────────────────────────
out_kossak = "E_KOSSAK_TRACEDIST_ALLSTATES_"*date_str*".h5"

h5open(out_kossak, "cw") do out
    h5open(kossak_fitted, "r") do fitted
        for γᵢ in γ
            println("\nKossak  γ = ", γᵢ)
            H = convert.(ComplexF64, read(fitted[γᵢ]["H"]))
            C = convert.(ComplexF64, read(fitted[γᵢ]["C"]))
            Hˢⁱᵈ = DenseOperator(basis, H)
            eff_L = LiPoSID.get_lindblad_operators(C, fᴷᴼᴺᴮ)
            ops   = [DenseOperator(basis, j) for j in eff_L]
            γ_grp = create_group(out, γᵢ)
            for state in all_states
                tₛ, ρₛ = LiPoSID.read_timeevolution(evol_data, state, γᵢ)
                ρₛ   = convert(Vector{Matrix{ComplexF64}}, ρₛ)
                tᵗˢᵗ = Float64.(tₛ)
                ρₒ   = DenseOperator(basis, ρₛ[1])
                tout, ρ_t = timeevolution.master(tᵗˢᵗ, ρₒ, Hˢⁱᵈ, ops)
                ρˢⁱᵈ = [x.data for x in ρ_t]
                @assert length(ρₛ) == length(ρˢⁱᵈ) "length mismatch γ=$γᵢ state=$state"
                td = LiPoSID.TrDist_series(ρₛ, ρˢⁱᵈ)
                sg = create_group(γ_grp, state)
                sg["TraceDist"] = convert.(Float64, td)
                sg["time"]      = tᵗˢᵗ
                println("  ", state, "  max=", round(maximum(td), digits=5))
            end
        end
    end
end
println("\nSaved: ", out_kossak)


Kossak  γ = 0.079477
  B1  max=0.05448
  B2  max=0.02813
  B3  max=0.06434
  B4  max=0.06422
  D1  max=0.06332
  D2  max=0.06337
  D3  max=0.05217
  D4  max=0.05227
  D5  max=0.06507
  D6  max=0.06507
  D7  max=0.0643
  D8  max=0.06435
  D9  max=0.05632
  D10  max=0.03377

Kossak  γ = 0.25133
  B1  max=0.0255
  B2  max=0.02464
  B3  max=0.02577
  B4  max=0.02571
  D1  max=0.02579
  D2  max=0.02579
  D3  max=0.02517
  D4  max=0.02522
  D5  max=0.02581
  D6  max=0.02581
  D7  max=0.02577
  D8  max=0.02576
  D9  max=0.02556
  D10  max=0.02474

Kossak  γ = 0.79477
  B1  max=0.0207
  B2  max=0.02033
  B3  max=0.02089
  B4  max=0.02091
  D1  max=0.02102
  D2  max=0.02111
  D3  max=0.02061
  D4  max=0.02061
  D5  max=0.02112
  D6  max=0.02114
  D7  max=0.02087
  D8  max=0.02089
  D9  max=0.02073
  D10  max=0.02037

Kossak  γ = 2.5133
  B1  max=0.01673
  B2  max=0.01629
  B3  max=0.01759
  B4  max=0.01831
  D1  max=0.01757
  D2  max=0.0188
  D3  max=0.01668
  D4  max=0.01771
  D5  max=0.01875

In [6]:
# ── Lindblad rescore ───────────────────────────────────────────────────────
out_lindblad = "E_LINDBLAD_TRACEDIST_ALLSTATES_"*date_str*".h5"

h5open(out_lindblad, "cw") do out
    h5open(lindblad_fitted, "r") do fitted
        for γᵢ in γ
            println("\nLindblad  γ = ", γᵢ)
            H  = convert.(ComplexF64, read(fitted[γᵢ]["H"]))
            J1 = convert.(ComplexF64, read(fitted[γᵢ]["J1"]))
            J2 = convert.(ComplexF64, read(fitted[γᵢ]["J2"]))
            J3 = convert.(ComplexF64, read(fitted[γᵢ]["J3"]))
            J4 = convert.(ComplexF64, read(fitted[γᵢ]["J4"]))
            Hˢⁱᵈ = DenseOperator(basis, H)
            ops  = [DenseOperator(basis, j) for j in (J1, J2, J3, J4)]
            γ_grp = create_group(out, γᵢ)
            for state in all_states
                tₛ, ρₛ = LiPoSID.read_timeevolution(evol_data, state, γᵢ)
                ρₛ   = convert(Vector{Matrix{ComplexF64}}, ρₛ)
                tᵗˢᵗ = Float64.(tₛ)
                ρₒ   = DenseOperator(basis, ρₛ[1])
                tout, ρ_t = timeevolution.master(tᵗˢᵗ, ρₒ, Hˢⁱᵈ, ops)
                ρˢⁱᵈ = [x.data for x in ρ_t]
                @assert length(ρₛ) == length(ρˢⁱᵈ) "length mismatch γ=$γᵢ state=$state"
                td = LiPoSID.TrDist_series(ρₛ, ρˢⁱᵈ)
                sg = create_group(γ_grp, state)
                sg["TraceDist"] = convert.(Float64, td)
                sg["time"]      = tᵗˢᵗ
                println("  ", state, "  max=", round(maximum(td), digits=5))
            end
        end
    end
end
println("\nSaved: ", out_lindblad)


Lindblad  γ = 0.079477
  B1  max=0.00074
  B2  max=0.00154
  B3  max=0.04364
  B4  max=0.04325
  D1  max=0.03551
  D2  max=0.03548
  D3  max=0.0355
  D4  max=0.03547
  D5  max=0.0404
  D6  max=0.0404
  D7  max=0.0436
  D8  max=0.04359
  D9  max=0.01558
  D10  max=0.01557

Lindblad  γ = 0.25133
  B1  max=0.00131
  B2  max=0.00126
  B3  max=0.00263
  B4  max=0.00253
  D1  max=0.00216
  D2  max=0.00222
  D3  max=0.00214
  D4  max=0.00221
  D5  max=0.0024
  D6  max=0.0024
  D7  max=0.00259
  D8  max=0.00264
  D9  max=0.00141
  D10  max=0.00136

Lindblad  γ = 0.79477
  B1  max=0.00376
  B2  max=0.00339
  B3  max=0.00776
  B4  max=0.00779
  D1  max=0.00642
  D2  max=0.00648
  D3  max=0.00648
  D4  max=0.00643
  D5  max=0.00727
  D6  max=0.0073
  D7  max=0.00781
  D8  max=0.00783
  D9  max=0.00402
  D10  max=0.00372

Lindblad  γ = 2.5133
  B1  max=0.00904
  B2  max=0.00711
  B3  max=0.02288
  B4  max=0.02372
  D1  max=0.01998
  D2  max=0.01769
  D3  max=0.02056
  D4  max=0.01773
  D5  max=0.